In [1]:
import os
from src.module.database import oracle_import, oracle_export, oracle_execute
#os.environ

In [2]:
marketplace_mp = oracle_import("""select id_, userid, p_date from toki.MONGO_MINIPROGRAMUSERLOGS partition(p_202605)
     where MINIPROGRAMID = '6821b668840548dbe178eacb'""")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-21 10:27:15.372 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 27 sec 


In [3]:
marketplace_mp["USERID"].unique().shape

(258604,)

In [4]:
marketplace_mp.tail()

,ID_,USERID,P_DATE
633632,6a1c5078982236d956e8040d,65afc92e5687245adfba3f93,20260531
633633,6a1c516cf081a0899db9ed8a,66d5712c98ce1d7839f29b29,20260531
633634,6a1c516ec54f01072f707b4a,655f4c21e22c5f4c8b76d1ad,20260531
633635,6a1c51d12091a9e894b4d1b2,65d39d6735ef84b89ee1f638,20260531
633636,6a1c528e6adb6e8a91d03110,60d5415041f18afccfbdc833,20260531


In [5]:
consumer_events = oracle_import("select * from toki.marketplace_consumer_EVENTS where p_date >='20260701'")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-21 10:27:39.826 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 24 sec 295 ms


In [6]:
customer_activities = oracle_import("select * from toki.marketplace_consumer_activities where p_date >='20260701'")

2026-07-21 10:29:27.724 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 47 sec 


In [7]:
customer_activities.head()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
0,6a442142805151b025d9ff1d,cart-events,"{'cartId': '63b572a5119256f621a0c935', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
1,6a442142ce31add3c3475288,cart-events,"{'cartId': '5fb9457f58786d2fc4e4a6b5', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
2,6a442142ce31add3c347528a,cart-events,"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
3,6a442142ce31add3c347528c,cart-events,"{'cartId': '68d27a73ca4a9a5563b54e15', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
4,6a442142805151b025d9ff1f,cart-events,"{'cartId': '68d51739282d4349b9bd7681', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701


In [42]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [9]:
import json
import ast

In [10]:
ast.literal_eval(customer_activities["ACTIVITYDATA"].values[0])

{'cartId': '63b572a5119256f621a0c935',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '692e313c49a5eecb319a3f58',
  'qty': 1,
  'available': True,
  '_id': '69a302381449fddd66765d91'},
 'cart': {'_id': '69a1c0191449fddd6670fcc5',
  'accountId': '63b572a5119256f621a0c935',
  'items': [{'productId': '692e313c49a5eecb319a3f58',
    'qty': 1,
    'available': True,
    '_id': '69a302381449fddd66765d91'},
   {'productId': '68febd519494859a95029a61',
    'qty': 1,
    'available': True,
    '_id': '69a54eac1449fddd66834326'},
   {'productId': '68febd4c9494859a95029a58',
    'qty': 1,
    'available': True,
    '_id': '69a54eec1449fddd66834389'}],
  'createdAt': '2026-02-27T16:02:33.965Z',
  'updatedAt': '2026-06-30T20:04:18.220Z'}}

In [11]:
from datetime import datetime, timedelta
snapshot_date = datetime.today().date()
snapshot_date = snapshot_date.strftime("%Y%m%d")

In [12]:
marketplace_products = oracle_import(f"select * from toki.marketplace_catalogue_products where SNAPSHOT_DATE >= to_date('{snapshot_date}', 'YYYYMMDD')")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-21 10:31:41.560 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 2 min 13 sec 


In [13]:
marketplace_products.head()

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
0,68477d5fe481aa03ae7a07ea,20ab2066-1ccf-55a7-975a-3e6e69e2f8dd,6822e76c24c12cc38727254a,672ad39ad05154ab2d517a58,1253163,SAMSUNG,10,3889990,3099970,20.31,...,['https://assets.nomin.mn/preview/4a/mn-crysta...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/6822e76c24c12cc38727254a,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...","[{'productId': '6822e76c24c12cc38727254a', '_i...",False,2025-06-10 08:33:35,2026-07-21 00:00:15,ARCHIVED,2026-07-21 00:03:20
1,68501465ef3ea8a746097598,Samsung Galaxy A34,650913f93f3ddf659787fe3a,672ad39ad05154ab2d517a5e,SM-A34-BLK-128GB,Samsung,0,1298000,1298000,0,...,['https://upload-web.toki.mn/uploads/leasing/r...,"[{'id': '6865110c060734bbf8b639b4', 'level': 0...",/650913f93f3ddf659787fe3a,"{'id': '6865110c060734bbf8b639b4', 'label': 'Г...","[{'productId': '650913f93f3ddf659787fe3a', '_i...",False,2025-06-16 20:56:05,2026-07-21 00:02:25,SETTLED,2026-07-21 00:03:20
2,68477d5fe481aa03ae7a07a1,202195,6822e76b24c12cc38727251a,672ad39ad05154ab2d517a58,1002055,XIAOMI,1,2499990,1599970,36,...,['https://assets.nomin.mn/preview/a8/xiaomi_mi...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/6822e76b24c12cc38727251a,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...","[{'productId': '6822e76b24c12cc38727251a', '_i...",False,2025-06-10 08:33:35,2025-09-22 18:01:41,ARCHIVED,2026-07-21 00:03:20
3,68650772c8e5d7eb7e8c32cf,AKG headphone(c type),663aeba2afc2032a592dbd39,672ad39ad05154ab2d517a5e,EO-IC100BWEGRU,Samsung,0,50000,50000,0,...,['https://upload-web.toki.mn/uploads/leasing/r...,"[{'id': '69895f8be783dbd39ed2240c', 'level': 0...",/663aeba2afc2032a592dbd39,"{'id': '69895f8be783dbd39ed2240c', 'label': 'Ч...","[{'productId': '663aeba2afc2032a592dbd39', '_i...",False,2025-07-02 18:18:26,2026-07-21 00:02:25,SETTLED,2026-07-21 00:03:20
4,68650778c8e5d7eb7e8c3318,Huawei Pura 70 Pro,667bc26a46c8f80fe35ed8d5,672ad39ad05154ab2d517a5e,HW-P70PRPWHT,Huawei,0,3588000,2438000,32.05,...,['https://upload-web.toki.mn/uploads/marketpla...,"[{'id': '6865110c060734bbf8b639b4', 'level': 0...",/667bc26a46c8f80fe35ed8d5,"{'id': '6865110c060734bbf8b639b4', 'label': 'Г...","[{'productId': '667bc26a46c8f80fe35ed8d5', '_i...",False,2025-07-02 18:18:32,2026-07-21 00:02:25,SETTLED,2026-07-21 00:03:20


In [14]:
productid = '69fc469bab34c8d11412ec79'
marketplace_products[marketplace_products["PRODUCTID"] == productid]

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
967,6a0b3d3b38713901eb69bc30,86280094-3bd8-58e3-a377-b7f51bd157a2,69fc469bab34c8d11412ec79,6912d444458a4a2f9ca10060,SF321LF0,ROWENTA,1,199900,199900,0,...,['https://cdnp.cody.mn/spree/images/3431242/la...,"[{'id': '69fb16a36712683b9c3f5b84', 'level': 0...",/69fc469bab34c8d11412ec79,"{'id': '69fb16a36712683b9c3f5b84', 'label': 'Ү...","[{'productId': '69fc469bab34c8d11412ec79', '_i...",False,2026-05-19 00:24:26,2026-07-21 00:00:21,ARCHIVED,2026-07-21 00:03:20


In [15]:
consumer_events.shape

(268957, 11)

In [16]:
consumer_events.tail()

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
268952,6a5e466f4229463e57f2888c,product_click,"{'productIds': ['68d3cd51d36b9be827b44e3f'], '...",61188d54d57f8d836d7e5a0e,oanTUT5MRw0IHeuzHjhKdxojE81e_jzv,2026-07-20T16:01:51.668Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home,2026-07-21 00:01:51,2026-07-21 00:01:51,20260721
268953,6a5e46764229463e57f2888e,taxon_click,{'taxon': {'label': 'Гар утас'}},6a5e457a55734a57204ed3c5,teUWvVvJg6gDdak2bp_YcBYT2zHlEGpg,2026-07-20T16:01:58.626Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home,2026-07-21 00:01:58,2026-07-21 00:01:58,20260721
268954,6a5e4677805151b025e97bfd,taxon_click,{'taxon': {'label': 'Гар утас'}},6150824e741a22cff4e4f195,lU9HkHo37SmhSxISUSXuBwk7nRJS4rs_,2026-07-20T16:01:59.226Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home,2026-07-21 00:01:59,2026-07-21 00:01:59,20260721
268955,6a5e4679420fe633e03d1b5e,taxon_click,{'taxon': {'label': 'Ухаалаг Цаг'}},6150824e741a22cff4e4f195,lU9HkHo37SmhSxISUSXuBwk7nRJS4rs_,2026-07-20T16:02:01.553Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/674429ce8f734...,2026-07-21 00:02:01,2026-07-21 00:02:01,20260721
268956,6a5e467c805151b025e97bff,product_click,"{'productIds': ['6a309f8cd46aca65f8084449'], '...",6108bca6edd53f3bd02dbf1c,vkpnkeK28TMMjokb3peKzBhJGQujTQ6a,2026-07-20T16:02:04.417Z,Mozilla/5.0 (Linux; Android 16; SM-A245N Build...,https://marketplace.toki.mn/home,2026-07-21 00:02:04,2026-07-21 00:02:04,20260721


In [17]:
consumer_events.head(2)

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
0,6a44ce39ce31add3c347e3d6,product_click,"{'productIds': ['69fc469bab34c8d11412ec79'], '...",66fbc5824e022311128232ae,jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy,2026-07-01T08:21:23.894Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/69f326e4f996d...,2026-07-01 16:22:17,2026-07-01 16:22:17,20260701
1,6a44ce3b420fe633e02e2e78,taxon_click,{'taxon': {'label': 'Гар утас'}},5ff870ee4f636263bd482270,2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9,2026-07-01T08:22:19.413Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like M...,https://marketplace.toki.mn/home,2026-07-01 16:22:19,2026-07-01 16:22:19,20260701


In [18]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,118629,118629,118629,117638,118629,118629,118629,118629,118629,118629
taxon_click,150328,146823,150328,149096,150328,150328,150328,150328,150328,150328


In [32]:
consumer_events[consumer_events["EVENTNAME"] == "taxon_click"]["EVENTVALUE"].unique()

<StringArray>
[                                                                                                                                '{'taxon': {'label': 'Гар утас'}}',
                                                                                                                                   '{'taxon': {'label': 'Зурагт'}}',
                                                                                                                          '{'taxon': {'label': 'Тренд технологи'}}',
                                                                                                                                '{'taxon': {'label': 'Гал тогоо'}}',
                                                                                                                                 '{'taxon': {'label': 'Гэр ахуй'}}',
                                                                                                                              '{'taxon': {'label': 'Мик, спикер'}

In [40]:
consumer_events[consumer_events["ACCOUNTID"] == '6a5e47214aeec353171ccaa0']

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE


In [20]:
consumer_events[consumer_events["SESSIONID"] == "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy"]["EVENTVALUE"].values

<StringArray>
['{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a309f8bd46aca65f808443d'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a052b7de66ad55426cacf70'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
                                                '{'taxon': {'label': 'Мик, спикер'}}',
         '{'productIds': ['69f9b680ce8b92727bb248a0'], 'taxon': {'label': 'Спикер'}}',
         '{'productIds': ['6a2bf50230ec49e38af99da7'], 'taxon': {'label': 'Спикер'}}',
                                             '{'taxon': {'label': 'Үсний хэрэгсэл'}}']
Length: 7, dtype: str

In [21]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [22]:
customer_activities["ACTIVITYDATA"].values[2]

"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}, 'cart': {'_id': '69a1d17e1449fddd6672c3e0', 'accountId': '69a1cf749d7bf25a7dcdf8ad', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}], 'createdAt': '2026-02-27T17:16:46.492Z', 'updatedAt': '2026-06-30T20:04:18.234Z'}}"

In [23]:
customer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [24]:
customer_activities.groupby(["ACTIVITYNAME"]).count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,2320028,2320028,2320028,2320028,2320028
limit-events,63845,63845,63845,63845,63845
order-events,11820,11820,11820,11820,11820
wishlist-events,1630,1630,1630,1630,1630


In [25]:
customer_activities[customer_activities["ACTIVITYNAME"]== 'cart-events'].head()["ACTIVITYDATA"].values[4]

"{'cartId': '68d51739282d4349b9bd7681', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}, 'cart': {'_id': '69a250a11449fddd6672e579', 'accountId': '68d51739282d4349b9bd7681', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}], 'createdAt': '2026-02-28T02:19:13.501Z', 'updatedAt': '2026-06-30T20:04:18.250Z'}}"

In [26]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [27]:
# there 4 events that could potentially be used to track user activity: "view_product"
# limit-events - user checked the lease limit
# order-events - user placed an order or completed an order
# wishlist-events - user added a product to their wishlist
# card-events added card or removed card, modified in the cards etc events,


In [28]:
from src.database import pgsql_import
master_catalog_profile = pgsql_import("select * from marketplace_catalog_data_extended_version3")

In [29]:
master_catalog_profile.head(2)

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,...,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9


In [46]:
master_catalog_profile.tail(2).to_json(orient="records")

'[{"carried_located_in":"Home","main_category":"Home Appliances","sub_category":"Vacuum Cleaners","product_category":"Cordless Vacuum Cleaners","exact_product_category":"Dyson V15s Detect Submarine Complete","manufacturer":"DYSON","generic_name":"Vacuum Cleaner","actual_product":"Dyson V15s Detect Submarine\\u2122 Complete","size":"4.1 kg","power_consumption":"","year":"0","specifications":"{\\"suction_power\\": \\"240 AW\\", \\"runtime\\": \\"60 minutes\\", \\"charging_time\\": \\"4.5 hours\\", \\"filter\\": \\"Whole-machine HEPA\\", \\"cleaning_type\\": \\"dry and wet\\", \\"dust_detection\\": \\"Dust Detect sensor\\", \\"display\\": \\"LCD\\", \\"head_type\\": \\"Submarine wet roller head\\", \\"light\\": \\"Fluffy Optic laser\\"}","sku":"ANTM-3193","connectivity":"{}","stock":"4","price":"4100000 MNT","dimensions":"","index":"shop_49_1782115641","product_id":"6a309f8bd46aca65f8084431","shop_name":"ANTMALL","discount":"{}","main_option":"{}","details":"Cordless vacuum cleaner with w

In [45]:
pd.set_option('display.max_columns', 100)

In [44]:
import pandas as pd